### Part 1: Environment Setup

In [11]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [12]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the server logs
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

# Give the server a few seconds to start up
time.sleep(5)

In [13]:
# Pull the lightweight multimodal vision model
!ollama pull llava

In [29]:
!ollama pull minicpm-v4.6:latest

### Part 2: Multimodal RAG with Your Uploaded Images

In [30]:
# Install required client SDKs
!pip install transformers pillow torch ollama numpy

In [31]:
import os
import torch
import numpy as np
from PIL import Image
import ollama
from transformers import CLIPProcessor, CLIPModel


In [32]:
# STEP 1: CONFIGURE THE REAL IMAGE STORE

def verify_and_load_images():
    """Validates that the user uploaded the correct images to Colab."""
    uploaded_files = ["imag1.png", "imag2.png"]

    for file_path in uploaded_files:
        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"Error: '{file_path}' not found in the Colab sidebar! "
                "Please drag and drop your files into the file explorer and run this again."
            )
    print("[1/4] Successfully located 'imag1.png' and 'imag2.png' in workspace.")
    return uploaded_files


In [33]:
# STEP 2: CLIP CROSS-MODAL INDEXING

def index_images_with_clip(image_paths):
    print("[2/4] Loading CLIP model to compute image vector profiles...")

    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    image_vectors = {}
    for path in image_paths:
        image = Image.open(path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")

        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
            # --- Extract the tensor for transformers v5.x ---
            if not isinstance(image_features, torch.Tensor):
                image_features = image_features.pooler_output
            # L2 Normalization makes dot product act exactly like cosine similarity
            image_features /= image_features.norm(dim=-1, keepdim=True)

        image_vectors[path] = image_features.numpy().flatten()
        print(f" -> Embedded {path} into a dense vector shape: {image_vectors[path].shape}")

    return model, processor, image_vectors


In [34]:
# STEP 3: TEXT-TO-IMAGE SEARCH (RETRIEVAL)

def retrieve_best_image(query, clip_model, clip_processor, image_index):
    print(f"\n[3/4] Running cross-modal vector search for query: '{query}'")

    inputs = clip_processor(text=[query], return_tensors="pt", padding=True)
    with torch.no_grad():
        text_features = clip_model.get_text_features(**inputs)
        # --- Extract the tensor for transformers v5.x ---
        if not isinstance(text_features, torch.Tensor):
            text_features = text_features.pooler_output
        text_features /= text_features.norm(dim=-1, keepdim=True)
    text_vector = text_features.numpy().flatten()

    best_score = -1
    best_match_path = None

    for path, img_vector in image_index.items():
        similarity = np.dot(text_vector, img_vector)
        print(f" -> Match Confidence with {path}: {similarity:.4f}")

        if similarity > best_score:
            best_score = similarity
            best_match_path = path

    print(f"Best Document Retrieved: {best_match_path}")
    return best_match_path


In [35]:
# STEP 4: MULTIMODAL INFERENCE VIA LOCAL OLLAMA

def ask_ollama_vision(image_path, prompt):
    print(f"[4/4] Sending retrieved context ({image_path}) to local 'minicpm' model...")

    try:
        response = ollama.chat(
            model='minicpm-v4.6:latest',
            messages=[{
                'role': 'user',
                'content': prompt,
                'images': [image_path]
            }]
        )
        print("\n================== LLM GENERATION RESULTS ==================")
        print(response['message']['content'].strip())
        print("=============================================================")
    except Exception as e:
        print(f"Ollama inference error encountered: {e}")


In [36]:
# EXECUTION PIPELINE

if __name__ == "__main__":
    # Validate the uploaded files
    target_images = verify_and_load_images()

    # Generate embeddings
    clip_model, clip_processor, encoded_image_index = index_images_with_clip(target_images)

    #
    # Query tailored for a visual pie chart breakdown
    #
    test_query = "Show me the colorful circular pie chart diagram detailing fruit distribution percentages"

    # Run Retrieval
    matched_image_path = retrieve_best_image(test_query, clip_model, clip_processor, encoded_image_index)

    # Run localized Ollama inspection prompt over the retrieved image
    ollama_prompt = "Look closely at this chart image. What is the specific percentage allocated to 'Bananas'? Answer with just the final number."
    ask_ollama_vision(matched_image_path, ollama_prompt)

[1/4] Successfully located 'imag1.png' and 'imag2.png' in workspace.
[2/4] Loading CLIP model to compute image vector profiles...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

 -> Embedded imag1.png into a dense vector shape: (512,)
 -> Embedded imag2.png into a dense vector shape: (512,)

[3/4] Running cross-modal vector search for query: 'Show me the colorful circular pie chart diagram detailing fruit distribution percentages'
 -> Match Confidence with imag1.png: 0.3840
 -> Match Confidence with imag2.png: 0.2966
Best Document Retrieved: imag1.png
[4/4] Sending retrieved context (imag1.png) to local 'minicpm' model...

================== LLM GENERATION RESULTS ==================
To find the specific percentage allocated to 'Bananas', we look at the segment labeled "Bananas" in the chart. The value there is 25%. Therefore, the final number is \boxed{25}.
